# Khmer AI-Text Detector — Notebook 2: Fine-tune XLM-RoBERTa (step 6)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cheamenglyzz-spec/khmer-ai-text-detector/blob/step2-ai-generation/notebooks/02_finetune_xlmr.ipynb)

**Goal:** fine-tune `xlm-roberta-base` to tell human-written Khmer text (label 0) from AI-generated text (label 1), then score it on val, test and the held-out generator (`test_ood`) with the same metrics as the step-5 baseline.

This notebook is only a driver. All the logic is in `src/finetune.py` in the repo, which it clones and runs.

**Needs**
- A **GPU runtime**: `Runtime → Change runtime type → T4 GPU`.
- The four split files `train.jsonl`, `val.jsonl`, `test.jsonl`, `test_ood.jsonl` (made locally by `src/split.py`). They are gitignored, so copy them to Google Drive at `MyDrive/khmer-ai-text-detector/splits/` first. If they are not there, the notebook tells you and you can upload them from your computer instead.

**Time:** roughly 5–10 minutes on a T4 for the current small splits (about 1,400 training texts), and 30–45 minutes for the full dataset. These are estimates. The training cell first downloads the model (about 1 GB).

**Outputs** go to Google Drive, so they survive the session: the model in `models/xlmr/`, plus `reports/step6_xlmr_metrics.json` and `reports/step6_xlmr_predictions.jsonl`. Copy the two report files into the repo's `reports/` folder afterwards.

**How to run:** `Runtime → Run all`. Change the settings in the next cell if needed.

In [ ]:
# --- settings ---------------------------------------------------------------
REPO_URL = "https://github.com/cheamenglyzz-spec/khmer-ai-text-detector"
BRANCH = "step2-ai-generation"  # use "main" once this branch is merged
REPO_DIR = "/content/khmer-ai-text-detector"

DRIVE_ROOT = "/content/drive/MyDrive/khmer-ai-text-detector"
SPLITS_DIR = f"{DRIVE_ROOT}/splits"
OUT_DIR = f"{DRIVE_ROOT}/models/xlmr"
METRICS = f"{DRIVE_ROOT}/reports/step6_xlmr_metrics.json"
PREDICTIONS = f"{DRIVE_ROOT}/reports/step6_xlmr_predictions.jsonl"
CHECKPOINT_DIR = "/content/xlmr_checkpoints"  # local disk: faster than Drive, and nothing temporary is left in Drive

EXTRA_ARGS = ""  # e.g. "--epochs 4 --lr 3e-5", or a quick test: "--limit 64 --max-steps 8 --max-length 128"

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), "No GPU found: Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Get the code

In [ ]:
import os

# The GitHub repo is private: add a token in Colab's Secrets panel (key icon on the left) named GITHUB_TOKEN
# (a fine-grained token with read access to this repo) and allow this notebook to use it.
# Without the secret the plain URL is used, which works once the repo is public.
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --quiet --depth 1 --branch {BRANCH} {clone_url} {REPO_DIR}
    !git -C {REPO_DIR} remote set-url origin {REPO_URL}  # don't keep the token in .git/config
assert os.path.isdir(f"{REPO_DIR}/src"), "Clone failed: set the GITHUB_TOKEN secret (private repo) and check BRANCH."
%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
# Colab already has torch, scikit-learn, pandas and datasets. pip only installs or upgrades what is missing or too old.
!pip install -q "transformers>=4.46" "accelerate>=1.1"

## 3. Find the data (Google Drive)

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

NEEDED = ["train", "val", "test", "test_ood"]
missing = [f"{n}.jsonl" for n in NEEDED if not (Path(SPLITS_DIR) / f"{n}.jsonl").exists()]
if missing:
    print(f"Missing in {SPLITS_DIR}: {', '.join(missing)}\nCopy them to that Drive folder and run this cell again, or run the upload cell below.")
else:
    for n in NEEDED:
        with open(Path(SPLITS_DIR) / f"{n}.jsonl", encoding="utf-8") as f:
            print(f"{n:<9}{sum(1 for line in f if line.strip()):>6} rows")

**Only if the cell above listed missing files:** this cell asks you to pick `train.jsonl`, `val.jsonl`, `test.jsonl` and `test_ood.jsonl` from your computer (it does nothing when Drive has them). They go to the Colab disk (not Drive), so upload them again in a new session.

In [ ]:
if missing:
    from google.colab import files

    SPLITS_DIR = "/content/splits"
    Path(SPLITS_DIR).mkdir(exist_ok=True)
    for name, data in files.upload().items():
        (Path(SPLITS_DIR) / name).write_bytes(data)
    print(sorted(p.name for p in Path(SPLITS_DIR).glob("*.jsonl")))
else:
    print(f"Using the splits in {SPLITS_DIR}")

## 4. Fine-tune and evaluate

In [ ]:
!python src/finetune.py --splits-dir "{SPLITS_DIR}" --out-dir "{OUT_DIR}" --checkpoint-dir "{CHECKPOINT_DIR}" --metrics "{METRICS}" --predictions "{PREDICTIONS}" {EXTRA_ARGS}

## 5. Results

In [ ]:
import json

import pandas as pd

report = json.loads(Path(METRICS).read_text(encoding="utf-8"))
train = report["training"]
print(f"Device {report['environment']['device']}, trained {train['train_seconds']:.0f}s, best val F1 {train['best_val_f1']} ({train['best_checkpoint']})\n")

print("Metrics (ai = positive class)")
display(pd.DataFrame(report["metrics"]).T[["n", "accuracy", "precision", "recall", "f1", "roc_auc"]])
for split in ("test", "test_ood"):
    print(f"{split}: recall per AI generator, accuracy on human texts per source")
    display(pd.DataFrame(report["breakdowns"][split]["ai_recall_by_generator"]).T)
    display(pd.DataFrame(report["breakdowns"][split]["human_accuracy_by_source"]).T)
print(f"Texts cut off at {report['config']['max_length']} tokens")
display(pd.DataFrame(report["truncation"]).T[["n", "truncated", "fraction", "tokens_median", "tokens_max"]])